In [4]:
import gzip
import json

CORPUS_PATH = r"C:\Users\ASUS\Downloads\c4-train.00000-of-01024-30K.json.gz"

with gzip.open(CORPUS_PATH, "rt", encoding="utf-8") as file:
    first_line = file.readline()

document = json.loads(first_line)

print(type(document))
print(document.keys())
print(document)

<class 'dict'>
dict_keys(['text', 'timestamp', 'url'])
{'text': 'Beginners BBQ Class Taking Place in Missoula!\nDo you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.\nHe will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat selection and trimming, plus smoker and fire information.\nThe cost to be in the class is $35 per person, and for spectators it is free. Included in the cost will be either a t-shirt or apron and you will be tasting samples of each meat that is prepared.', 'timestamp': '2019-04-25T12:57:54Z', 'url': 'https://klyq.com/beginners-bbq-class-taking-place-in-missoula/'}


In [5]:
documents = []

with gzip.open(CORPUS_PATH, "rt", encoding="utf-8") as file:
    for line in file:
        document = json.loads(line)
        documents.append(document["text"])

print("Số documents:", len(documents))
print("Document đầu tiên:")
print(documents[0][:500])

Số documents: 30000
Document đầu tiên:
Beginners BBQ Class Taking Place in Missoula!
Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.
He will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat select


In [6]:
from implementation import tokenize

tokenized_documents = [
    tokenize(document)
    for document in documents
]

print("Số documents:", len(tokenized_documents))
print("Document đầu tiên sau khi tokenize:")
print(tokenized_documents[0][:30])

Số documents: 30000
Document đầu tiên sau khi tokenize:
['beginners', 'bbq', 'class', 'taking', 'place', 'in', 'missoula', 'do', 'you', 'want', 'to', 'get', 'better', 'at', 'making', 'delicious', 'bbq', 'you', 'will', 'have', 'the', 'opportunity', 'put', 'this', 'on', 'your', 'calendar', 'now', 'thursday', 'september']


In [7]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

count_matrix = vectorizer.fit_transform(documents)

vocabulary = vectorizer.get_feature_names_out()

print("Số documents:", count_matrix.shape[0])
print("Số terms:", count_matrix.shape[1])
print("Shape của Count Matrix:", count_matrix.shape)

Số documents: 30000
Số terms: 193540
Shape của Count Matrix: (30000, 193540)


In [8]:
total_elements = count_matrix.shape[0] * count_matrix.shape[1]
non_zero_elements = count_matrix.nnz

sparsity = 1 - (non_zero_elements / total_elements)

print("Tổng số phần tử:", total_elements)
print("Số phần tử khác 0:", non_zero_elements)
print("Sparsity:", sparsity)
print("Sparsity (%):", sparsity * 100)

Tổng số phần tử: 5806200000
Số phần tử khác 0: 4985822
Sparsity: 0.9991412934449382
Sparsity (%): 99.91412934449382


In [9]:
import numpy as np

document_frequency = np.asarray(
    (count_matrix > 0).sum(axis=0)
).ravel()

top_df_indices = np.argsort(
    document_frequency
)[-20:][::-1]

print("Top 20 terms theo Document Frequency:")
print()

for rank, index in enumerate(top_df_indices, start=1):
    print(
        f"{rank:2}. {vocabulary[index]:20} "
        f"DF = {document_frequency[index]}"
    )

Top 20 terms theo Document Frequency:

 1. the                  DF = 27893
 2. and                  DF = 27423
 3. to                   DF = 26689
 4. of                   DF = 26031
 5. in                   DF = 25224
 6. for                  DF = 23651
 7. is                   DF = 22739
 8. with                 DF = 21405
 9. on                   DF = 20262
10. that                 DF = 18370
11. this                 DF = 17840
12. are                  DF = 17594
13. it                   DF = 17168
14. as                   DF = 16467
15. at                   DF = 16347
16. from                 DF = 16316
17. be                   DF = 16153
18. you                  DF = 16094
19. by                   DF = 15123
20. have                 DF = 14852


In [10]:
import numpy as np

number_of_documents = count_matrix.shape[0]

idf = np.zeros(len(vocabulary))

for i, df in enumerate(document_frequency):
    if df > 0:
        idf[i] = np.log(
            number_of_documents / df
        )

print("Số lượng IDF:", len(idf))
print("IDF nhỏ nhất:", idf.min())
print("IDF lớn nhất:", idf.max())

Số lượng IDF: 193540
IDF nhỏ nhất: 0.07282162037186433
IDF lớn nhất: 10.308952660644293


In [11]:
top_idf_indices = np.argsort(idf)[-20:][::-1]

print("Top 20 terms theo IDF:")
print()

for rank, index in enumerate(top_idf_indices, start=1):
    print(
        f"{rank:2}. {vocabulary[index]:20} "
        f"IDF = {idf[index]:.6f} "
        f"DF = {document_frequency[index]}"
    )

Top 20 terms theo IDF:

 1. 00000                IDF = 10.308953 DF = 1
 2. 00003                IDF = 10.308953 DF = 1
 3. 000040               IDF = 10.308953 DF = 1
 4. 00005                IDF = 10.308953 DF = 1
 5. 0000856166           IDF = 10.308953 DF = 1
 6. 0001042              IDF = 10.308953 DF = 1
 7. 000116               IDF = 10.308953 DF = 1
 8. 00012                IDF = 10.308953 DF = 1
 9. 00015                IDF = 10.308953 DF = 1
10. 00016                IDF = 10.308953 DF = 1
11. 000165101            IDF = 10.308953 DF = 1
12. 0002                 IDF = 10.308953 DF = 1
13. 00022                IDF = 10.308953 DF = 1
14. 000226               IDF = 10.308953 DF = 1
15. 000281               IDF = 10.308953 DF = 1
16. 확인하게                 IDF = 10.308953 DF = 1
17. 환원되지                 IDF = 10.308953 DF = 1
18. 활동                   IDF = 10.308953 DF = 1
19. 활동에                  IDF = 10.308953 DF = 1
20. 활동을                  IDF = 10.308953 DF = 1


In [12]:
tf_matrix = count_matrix.multiply(
    1 / count_matrix.sum(axis=1)
)

tfidf_matrix = tf_matrix.multiply(idf)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Số phần tử khác 0:", tfidf_matrix.nnz)

TF-IDF matrix shape: (30000, 193540)
Số phần tử khác 0: 4985822


In [13]:
doc_index = 0

tfidf_row = tfidf_matrix.getrow(doc_index)

top_tfidf_indices = np.argsort(
    tfidf_row.toarray().ravel()
)[-20:][::-1]

print(f"Top 20 TF-IDF terms của document {doc_index}:")
print()

for rank, index in enumerate(top_tfidf_indices, start=1):
    value = tfidf_row[0, index]

    print(
        f"{rank:2}. {vocabulary[index]:20} "
        f"TF-IDF = {value:.6f}"
    )

Top 20 TF-IDF terms của document 0:

 1. bbq                  TF-IDF = 0.184033
 2. class                TF-IDF = 0.097049
 3. balay                TF-IDF = 0.081173
 4. kcbs                 TF-IDF = 0.075715
 5. meat                 TF-IDF = 0.074776
 6. lonestar             TF-IDF = 0.072522
 7. missoula             TF-IDF = 0.063042
 8. apron                TF-IDF = 0.058414
 9. smoker               TF-IDF = 0.057988
10. timelines            TF-IDF = 0.054134
11. trimming             TF-IDF = 0.053178
12. spectators           TF-IDF = 0.052127
13. rangers              TF-IDF = 0.050529
14. 22nd                 TF-IDF = 0.047948
15. beginner             TF-IDF = 0.046868
16. beginners            TF-IDF = 0.046474
17. culinary             TF-IDF = 0.046285
18. tasting              TF-IDF = 0.044019
19. cost                 TF-IDF = 0.043857
20. tony                 TF-IDF = 0.042968


In [15]:
print("=" * 60)
print("PART D - TF-IDF CORPUS SUMMARY")

print(f"Number of documents: {count_matrix.shape[0]:,}")
print(f"Vocabulary size: {count_matrix.shape[1]:,}")
print(f"Count matrix shape: {count_matrix.shape}")
print(f"Non-zero elements: {count_matrix.nnz:,}")
print(f"Sparsity: {sparsity * 100:.4f}%")
print(f"Minimum IDF: {idf.min():.6f}")
print(f"Maximum IDF: {idf.max():.6f}")
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

PART D - TF-IDF CORPUS SUMMARY
Number of documents: 30,000
Vocabulary size: 193,540
Count matrix shape: (30000, 193540)
Non-zero elements: 4,985,822
Sparsity: 99.9141%
Minimum IDF: 0.072822
Maximum IDF: 10.308953
TF-IDF matrix shape: (30000, 193540)


In [16]:
# ============================================================
# PART F - PIPELINE A: MINIMAL

pipeline_a_tokens = [
    tokenize(document)
    for document in documents
]

avg_tokens_a = sum(
    len(tokens)
    for tokens in pipeline_a_tokens
) / len(pipeline_a_tokens)

vocab_a = set()

for tokens in pipeline_a_tokens:
    vocab_a.update(tokens)

print("Pipeline A - Minimal")
print("Vocabulary size:", len(vocab_a))
print("Average tokens/document:", avg_tokens_a)

Pipeline A - Minimal
----------------------------------------
Vocabulary size: 177185
Average tokens/document: 362.5350666666667


In [17]:
import re

def normalize_punctuation(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


pipeline_b_tokens = [
    tokenize(normalize_punctuation(document))
    for document in documents
]

avg_tokens_b = sum(
    len(tokens)
    for tokens in pipeline_b_tokens
) / len(pipeline_b_tokens)

vocab_b = set()

for tokens in pipeline_b_tokens:
    vocab_b.update(tokens)

print("Pipeline B - Before Stopword Handling")
print("Vocabulary size:", len(vocab_b))
print("Average tokens/document:", avg_tokens_b)

Pipeline B - Before Stopword Handling
Vocabulary size: 177185
Average tokens/document: 362.5350666666667


In [19]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

english_stopwords = set(ENGLISH_STOP_WORDS)

print("Số stopwords:", len(english_stopwords))
print("Một số stopwords:", list(english_stopwords)[:20])

Số stopwords: 318
Một số stopwords: ['go', 'on', 'no', 'put', 'him', 'afterwards', 'an', 'done', 'by', 'whom', 'onto', 'latter', 'was', 'noone', 'through', 'somehow', 'mine', 'are', 'whither', 'cannot']


In [20]:
pipeline_b_tokens = []

for document in documents:
    normalized_text = normalize_punctuation(document)
    tokens = tokenize(normalized_text)

    filtered_tokens = [
        token
        for token in tokens
        if token not in english_stopwords
    ]

    pipeline_b_tokens.append(filtered_tokens)


avg_tokens_b = sum(
    len(tokens)
    for tokens in pipeline_b_tokens
) / len(pipeline_b_tokens)

vocab_b = set()

for tokens in pipeline_b_tokens:
    vocab_b.update(tokens)

print("Pipeline B - Normalized")
print("Vocabulary size:", len(vocab_b))
print("Average tokens/document:", avg_tokens_b)

Pipeline B - Normalized
Vocabulary size: 176869
Average tokens/document: 190.15983333333332


In [23]:
from collections import Counter, defaultdict

class SimpleBPE:

    END = "</w>"

    def __init__(self, num_merges=300):
        self.num_merges = num_merges
        self.merges = []
        self.cache = {}

    def word_to_symbols(self, word):
        return list(word) + [self.END]

    def train(self, word_freq):
        vocab = {
            tuple(self.word_to_symbols(word)): freq
            for word, freq in word_freq.items()
        }

        for _ in range(self.num_merges):
            pair_counts = defaultdict(int)

            for symbols, freq in vocab.items():
                for i in range(len(symbols) - 1):
                    pair = (
                        symbols[i],
                        symbols[i + 1]
                    )
                    pair_counts[pair] += freq

            if not pair_counts:
                break

            best_pair = max(
                pair_counts,
                key=pair_counts.get
            )

            if pair_counts[best_pair] < 2:
                break

            self.merges.append(best_pair)

            a, b = best_pair
            merged_symbol = a + b

            new_vocab = {}

            for symbols, freq in vocab.items():
                new_symbols = []
                i = 0

                while i < len(symbols):
                    if (
                            i < len(symbols) - 1
                            and symbols[i] == a
                            and symbols[i + 1] == b
                    ):
                        new_symbols.append(merged_symbol)
                        i += 2
                    else:
                        new_symbols.append(symbols[i])
                        i += 1

                key = tuple(new_symbols)
                new_vocab[key] = (
                        new_vocab.get(key, 0) + freq
                )

            vocab = new_vocab

    def encode_word(self, word):
        if word in self.cache:
            return self.cache[word]

        symbols = self.word_to_symbols(word)

        for a, b in self.merges:
            merged_symbol = a + b
            new_symbols = []
            i = 0

            while i < len(symbols):
                if (
                        i < len(symbols) - 1
                        and symbols[i] == a
                        and symbols[i + 1] == b
                ):
                    new_symbols.append(merged_symbol)
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1

            symbols = new_symbols

        self.cache[word] = symbols

        return symbols

In [24]:
word_freq = Counter()

for document in documents:
    tokens = tokenize(document)
    word_freq.update(tokens)

top_words = dict(
    word_freq.most_common(5000)
)

print("Tổng số từ khác nhau:", len(word_freq))
print("Số từ dùng để train BPE:", len(top_words))
print("5 từ phổ biến nhất:")

for word, frequency in list(top_words.items())[:5]:
    print(word, "->", frequency)

Tổng số từ khác nhau: 177185
Số từ dùng để train BPE: 5000
5 từ phổ biến nhất:
the -> 564551
and -> 323974
to -> 307757
of -> 271669
a -> 245232


In [25]:
bpe = SimpleBPE(num_merges=300)

bpe.train(top_words)

print("Số merge rules đã học:", len(bpe.merges))
print("10 merge rules đầu tiên:")

for merge in bpe.merges[:10]:
    print(merge)

Số merge rules đã học: 300
10 merge rules đầu tiên:
('e', '</w>')
('t', 'h')
('s', '</w>')
('t', '</w>')
('d', '</w>')
('i', 'n')
('a', 'n')
('r', '</w>')
('th', 'e</w>')
('y', '</w>')


In [27]:
def pipeline_C(text):
    text = text.lower()
    text = normalize_punctuation(text)

    words = tokenize(text)

    subwords = []

    for word in words:
        subwords.extend(
            bpe.encode_word(word)
        )

    return subwords

print("Pipeline C test:")
print(pipeline_C("The transformer model is powerful"))

Pipeline C test:
['the</w>', 'tr', 'an', 's', 'for', 'm', 'er</w>', 'mo', 'de', 'l</w>', 'is</w>', 'p', 'ow', 'er', 'fu', 'l</w>']


In [28]:
pipeline_c_tokens = [
    pipeline_C(document)
    for document in documents
]

vocab_c = set()
total_tokens_c = 0

for tokens in pipeline_c_tokens:
    vocab_c.update(tokens)
    total_tokens_c += len(tokens)

avg_tokens_c = (
        total_tokens_c / len(documents)
)

print("Pipeline C - Extended")
print("-" * 40)
print("Vocabulary size:", len(vocab_c))
print("Average tokens/document:", avg_tokens_c)

Pipeline C - Extended
----------------------------------------
Vocabulary size: 2494
Average tokens/document: 898.9904


In [29]:
def calculate_sparsity(token_lists):
    vocab = set()

    for tokens in token_lists:
        vocab.update(tokens)

    vocab = sorted(vocab)

    term_to_index = {
        term: i
        for i, term in enumerate(vocab)
    }

    non_zero = 0

    for tokens in token_lists:
        unique_tokens = set(tokens)

        for token in unique_tokens:
            if token in term_to_index:
                non_zero += 1

    total_elements = (
            len(token_lists) * len(vocab)
    )

    sparsity = 1 - (
            non_zero / total_elements
    )

    return sparsity, len(vocab), non_zero


sparsity_a, _, non_zero_a = calculate_sparsity(
    pipeline_a_tokens
)

sparsity_b, _, non_zero_b = calculate_sparsity(
    pipeline_b_tokens
)

sparsity_c, _, non_zero_c = calculate_sparsity(
    pipeline_c_tokens
)

print("Matrix sparsity")
print("-" * 40)
print(f"Pipeline A: {sparsity_a:.4%}")
print(f"Pipeline B: {sparsity_b:.4%}")
print(f"Pipeline C: {sparsity_c:.4%}")

Matrix sparsity
----------------------------------------
Pipeline A: 99.9067%
Pipeline B: 99.9321%
Pipeline C: 93.3091%


In [30]:
import random

random.seed(42)

indices = list(range(len(documents)))
random.shuffle(indices)

split_point = int(len(indices) * 0.8)

train_indices = indices[:split_point]
test_indices = indices[split_point:]

train_documents = [
    documents[i]
    for i in train_indices
]

test_documents = [
    documents[i]
    for i in test_indices
]

print("Train documents:", len(train_documents))
print("Test documents:", len(test_documents))

Train documents: 24000
Test documents: 6000


In [32]:
def pipeline_A(text):
    return tokenize(text)


def pipeline_B(text):
    text = normalize_punctuation(text)
    tokens = tokenize(text)

    return [
        token
        for token in tokens
        if token not in english_stopwords
    ]

In [33]:
def calculate_oov_rate(train_tokens, test_tokens):
    train_vocab = set()

    for tokens in train_tokens:
        train_vocab.update(tokens)

    total_tokens = 0
    oov_tokens = 0

    for tokens in test_tokens:
        for token in tokens:
            total_tokens += 1

            if token not in train_vocab:
                oov_tokens += 1

    return oov_tokens / total_tokens


train_a = [pipeline_A(document) for document in train_documents]
test_a = [pipeline_A(document) for document in test_documents]

train_b = [pipeline_B(document) for document in train_documents]
test_b = [pipeline_B(document) for document in test_documents]

train_c = [pipeline_C(document) for document in train_documents]
test_c = [pipeline_C(document) for document in test_documents]


oov_a = calculate_oov_rate(train_a, test_a)
oov_b = calculate_oov_rate(train_b, test_b)
oov_c = calculate_oov_rate(train_c, test_c)


print("OOV rate")
print("-" * 40)
print(f"Pipeline A: {oov_a:.4%}")
print(f"Pipeline B: {oov_b:.4%}")
print(f"Pipeline C: {oov_c:.4%}")

OOV rate
----------------------------------------
Pipeline A: 1.7273%
Pipeline B: 3.2887%
Pipeline C: 0.0058%


In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import time

query_documents = test_documents[:30]

def tokens_to_text(token_lists):
    return [" ".join(tokens) for tokens in token_lists]

In [36]:
def measure_search_performance(train_tokens, query_tokens):
    train_texts = tokens_to_text(train_tokens)
    query_texts = tokens_to_text(query_tokens)

    vectorizer = TfidfVectorizer(
        lowercase=False,
        token_pattern=r"(?u)\b\w+\b"
    )

    train_matrix = vectorizer.fit_transform(train_texts)

    start = time.perf_counter()

    query_matrix = vectorizer.transform(query_texts)
    similarities = cosine_similarity(query_matrix, train_matrix)

    for row in similarities:
        top_indices = row.argsort()[-5:][::-1]

    elapsed = time.perf_counter() - start

    return elapsed, train_matrix.shape

In [37]:
query_a = [pipeline_A(document) for document in query_documents]
query_b = [pipeline_B(document) for document in query_documents]
query_c = [pipeline_C(document) for document in query_documents]

time_a, shape_a = measure_search_performance(train_a, query_a)
time_b, shape_b = measure_search_performance(train_b, query_b)
time_c, shape_c = measure_search_performance(train_c, query_c)

print("Search performance")
print("-" * 50)

print(f"Pipeline A: {time_a:.4f} seconds, matrix = {shape_a}")
print(f"Pipeline B: {time_b:.4f} seconds, matrix = {shape_b}")
print(f"Pipeline C: {time_c:.4f} seconds, matrix = {shape_c}")

Search performance
--------------------------------------------------
Pipeline A: 0.1144 seconds, matrix = (24000, 155838)
Pipeline B: 0.0807 seconds, matrix = (24000, 155523)
Pipeline C: 0.1520 seconds, matrix = (24000, 2241)


In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

search_vectorizer = TfidfVectorizer()

search_tfidf = search_vectorizer.fit_transform(documents)

print("Number of documents:", search_tfidf.shape[0])
print("Vocabulary size:", search_tfidf.shape[1])
print("TF-IDF matrix shape:", search_tfidf.shape)

Number of documents: 30000
Vocabulary size: 193540
TF-IDF matrix shape: (30000, 193540)


In [39]:
def search_documents(query, top_k=5):
    query_vector = search_vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        search_tfidf
    )[0]

    top_indices = scores.argsort()[-top_k:][::-1]

    results = []

    for rank, index in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "document_id": index,
            "similarity": scores[index],
            "preview": documents[index][:300]
        })

    return results

In [40]:
results = search_documents("medical image classification", top_k=5)

for result in results:
    print(f"Rank: {result['rank']}")
    print(f"Document ID: {result['document_id']}")
    print(f"Similarity: {result['similarity']:.6f}")
    print(f"Preview: {result['preview']}")
    print("-" * 80)

Rank: 1
Document ID: 18971
Similarity: 0.400030
Preview: The new RTS Environmental Classification system (RTS GLT) is designed for parties who are commissioning construction projects and who want to build in an environmentally responsible manner. The environmental classification system was developed for Finland, and it takes into account Finnish condition
--------------------------------------------------------------------------------
Rank: 2
Document ID: 8527
Similarity: 0.349998
Preview: History of maize classification. How races used in classification. Geographical distribution. Existing races of maize in Mexico.
--------------------------------------------------------------------------------
Rank: 3
Document ID: 19908
Similarity: 0.253427
Preview: Download League Of Legends Wallpapers in high-quality for your desktop and smart-phone in wide-screen and HD resolution.
Right click on image, and select “Save Image as League Of Legends Wallpapers” to download image to your desktop, lapt

In [41]:
queries = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing"
]

for query in queries:
    print("=" * 100)
    print("QUERY:", query)
    print("=" * 100)

    results = search_documents(query, top_k=5)

    for result in results:
        print(
            f"Rank {result['rank']} | "
            f"Document ID {result['document_id']} | "
            f"Similarity {result['similarity']:.6f}"
        )
        print(f"Preview: {result['preview']}")
        print("-" * 100)

Rank 1 | Document ID 27936 | Similarity 0.471849
Preview: hi, I am having problems with transformer / circuit board om my Hobby 720 uml model 2003.
In view of the problems, can I bypass the transformer/circuit board ??
----------------------------------------------------------------------------------------------------
Rank 2 | Document ID 25428 | Similarity 0.279103
Preview: Note: If you're on an iPhone, you cannot change the language of Facebook through the mobile app. Instead, Facebook uses whatever language your phone is set up to use, so to change it you have to pick a different language for your entire phone. Do this through the Settings app via General > Language 
----------------------------------------------------------------------------------------------------
Rank 3 | Document ID 4075 | Similarity 0.222687
Preview: Looking for Spanish language instructor to improve my reading, writing and speaking skills. We require the tutor to travel to our location at Gwal Pahari. A certif

In [42]:
for query in queries:
    results = search_documents(query, top_k=5)

    scores = [result["similarity"] for result in results]

    assert scores == sorted(scores, reverse=True)

print("Ranking check: PASS")

Ranking check: PASS


In [43]:
evaluation_queries = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing",
    "computer vision",
    "machine learning"
]

In [45]:
evaluation_results = {}

for query in evaluation_queries:
    results = search_documents(query, top_k=5)
    evaluation_results[query] = results

    print("=" * 100)
    print("QUERY:", query)

    for result in results:
        print(
            f"Rank {result['rank']} | "
            f"Doc ID {result['document_id']} | "
            f"Similarity {result['similarity']:.6f}"
        )
        print(f"Preview: {result['preview'][:200]}")
        print()

QUERY: natural language processing
Rank 1 | Doc ID 25428 | Similarity 0.362266
Preview: Note: If you're on an iPhone, you cannot change the language of Facebook through the mobile app. Instead, Facebook uses whatever language your phone is set up to use, so to change it you have to pick 

Rank 2 | Doc ID 8705 | Similarity 0.341789
Preview: These regulations may be called the Food Safety and Standards (Food Products Standards and Food Additives) Amendment Regulations, 2019. They shall come into force on the date of their publication in t

Rank 3 | Doc ID 4075 | Similarity 0.289040
Preview: Looking for Spanish language instructor to improve my reading, writing and speaking skills. We require the tutor to travel to our location at Gwal Pahari. A certification or degree in Spanish Language

Rank 4 | Doc ID 5699 | Similarity 0.282915
Preview: Truth-value gaps in natural language.
Waldo, James Hewins, "Truth-value gaps in natural language." (1980). Doctoral Dissertations 1896 - February 2014

In [49]:
evaluation_queries = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing",
    "computer vision"
]

In [51]:
K = 5

evaluation_results = {}

for query in evaluation_queries:
    results = search_documents(query, top_k=K)
    evaluation_results[query] = results

    print("QUERY:", query)
    print("=" * 100)

    for result in results:
        print(
            f"Rank {result['rank']} | "
            f"Document ID {result['document_id']} | "
            f"Similarity {result['similarity']:.6f}"
        )
        print(f"Preview: {result['preview']}")
        print("-" * 100)

QUERY: deep learning healthcare
Rank 1 | Document ID 9252 | Similarity 0.278174
Preview: SAN DIEGO AND WASHINGTON, D.C. – Sept. 5, 2018 – West Health, a family of nonpartisan, nonprofit organizations focused on healthcare research, policy and philanthropy, has appointed Cristina Boccuti, MA, MPP, formerly of the Kaiser Family Foundation, as director of health policy at the Gary and Mary
----------------------------------------------------------------------------------------------------
Rank 2 | Document ID 6123 | Similarity 0.275276
Preview: With today’s advancement in technology, it is becoming more convenient than ever before to earn a Bachelor’s degree in a health-related field online. Some specializations, such as nutrition or public health, can be studied entirely online. Other online degrees, such as Bachelor in Dentistry or Bache
----------------------------------------------------------------------------------------------------
Rank 3 | Document ID 11119 | Similarity 0.274719
P

In [57]:
relevance_labels = {
    "medical image classification": {17794},
    "transformer language model": {25428},
    "deep learning healthcare": {6123},
    "natural language processing": set(),
    "computer vision": {15116}
}

In [59]:
K = 5

def precision_at_k(retrieved_ids, relevant_ids, k):
    top_k = retrieved_ids[:k]
    relevant_retrieved = sum(doc_id in relevant_ids for doc_id in top_k)
    return relevant_retrieved / k


def recall_at_k(retrieved_ids, relevant_ids, k):
    if not relevant_ids:
        return None

    top_k = retrieved_ids[:k]
    relevant_retrieved = sum(doc_id in relevant_ids for doc_id in top_k)
    return relevant_retrieved / len(relevant_ids)


def first_relevant_rank(full_ranking, relevant_ids):
    for rank, doc_id in enumerate(full_ranking, start=1):
        if doc_id in relevant_ids:
            return rank
    return None


def mean_reciprocal_rank(reciprocal_ranks):
    if not reciprocal_ranks:
        return 0.0

    return sum(reciprocal_ranks) / len(reciprocal_ranks)

In [61]:
from sklearn.metrics.pairwise import cosine_similarity

all_precision = []
all_recall = []
all_rr = []

for query in evaluation_queries:
    relevant_ids = relevance_labels[query]

    results = search_documents(query, top_k=K)
    retrieved_ids = [result["document_id"] for result in results]

    query_vector = search_vectorizer.transform([query])
    similarities = cosine_similarity(
        query_vector,
        search_tfidf
    )[0]

    full_ranking = similarities.argsort()[::-1].tolist()

    p_at_5 = precision_at_k(retrieved_ids, relevant_ids, K)
    r_at_5 = recall_at_k(retrieved_ids, relevant_ids, K)

    rank = first_relevant_rank(full_ranking, relevant_ids)
    rr = 1 / rank if rank is not None else 0.0

    all_precision.append(p_at_5)
    all_recall.append(r_at_5)
    all_rr.append(rr)

    print("=" * 80)
    print("Query:", query)
    print("Relevant documents:", relevant_ids)
    print("Retrieved:", retrieved_ids)
    print(f"Precision@5: {p_at_5:.4f}")
    print(f"Recall@5:    {'N/A' if r_at_5 is None else f'{r_at_5:.4f}'}")
    print(f"First relevant rank: {rank}")
    print(f"Reciprocal Rank: {rr:.4f}")

valid_recalls = [r for r in all_recall if r is not None]

mean_precision = sum(all_precision) / len(all_precision)
mean_recall = sum(valid_recalls) / len(valid_recalls)
mrr = mean_reciprocal_rank(all_rr)

print("=" * 80)
print("FINAL EVALUATION")
print(f"Mean Precision@5: {mean_precision:.4f}")
print(f"Mean Recall@5:    {mean_recall:.4f}")
print(f"MRR:              {mrr:.4f}")

Query: computer vision
Relevant documents: {15116}
Retrieved: [np.int64(3820), np.int64(21344), np.int64(15116), np.int64(28397), np.int64(17379)]
Precision@5: 0.2000
Recall@5:    1.0000
First relevant rank: 3
Reciprocal Rank: 0.3333
FINAL EVALUATION
Mean Precision@5: 0.1600
Mean Recall@5:    1.0000
MRR:              0.3167


In [62]:
error_analysis_queries = [
    "transformer language model",
    "deep learning healthcare",
    "medical image classification",
    "natural language processing"
]

for query in error_analysis_queries:
    print("=" * 100)
    print("QUERY:", query)
    print("EXPECTED RELEVANT:", relevance_labels[query])
    print("=" * 100)

    results = search_documents(query, top_k=5)

    for result in results:
        print(
            f"Rank {result['rank']} | "
            f"Document ID {result['document_id']} | "
            f"Similarity {result['similarity']:.6f}"
        )
        print("Preview:", result["preview"])
        print("-" * 100)

Rank 1 | Document ID 25428 | Similarity 0.362266
Preview: Note: If you're on an iPhone, you cannot change the language of Facebook through the mobile app. Instead, Facebook uses whatever language your phone is set up to use, so to change it you have to pick a different language for your entire phone. Do this through the Settings app via General > Language 
----------------------------------------------------------------------------------------------------
Rank 2 | Document ID 8705 | Similarity 0.341789
Preview: These regulations may be called the Food Safety and Standards (Food Products Standards and Food Additives) Amendment Regulations, 2019. They shall come into force on the date of their publication in the Official Gazette.
In the Food Safety and Standards (Food Products Standards and Food Additives) r
----------------------------------------------------------------------------------------------------
Rank 3 | Document ID 4075 | Similarity 0.289040
Preview: Looking for Spanish lang